# VerseCast — fine-tune Whisper `small.en` with LoRA

Trains on **your own sermon dataset** packed by `scripts/07_pack_dataset.py`.

**Privacy note (read before running):** uploading `dataset.tar.gz` in cell 3 sends your sermon audio to *your own* Colab session (Google infrastructure), where it is deleted when the runtime is recycled. If that is not acceptable, rent a GPU box (e.g. Lambda/RunPod) and run these same cells there instead.

A free T4 handles a 3–6 h dataset in roughly **1–3 hours**. Runtime → Change runtime type → **T4 GPU** before starting.

In [ ]:
!pip install -q "transformers>=4.44" datasets peft accelerate evaluate jiwer soundfile librosa audiomentations
import torch
assert torch.cuda.is_available(), 'No GPU — Runtime → Change runtime type → T4 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
# Upload dataset.tar.gz (produced by scripts/07_pack_dataset.py)
from google.colab import files
up = files.upload()
assert 'dataset.tar.gz' in up, 'upload the dataset.tar.gz produced by 07_pack_dataset.py'
!mkdir -p /content/work && tar -xzf dataset.tar.gz -C /content/work
!ls /content/work/dataset/manifests

In [ ]:
import json, numpy as np, soundfile as sf
from pathlib import Path
from datasets import Dataset
from transformers import WhisperProcessor

BASE = 'openai/whisper-small.en'
WORK = Path('/content/work/dataset')
SR = 16000
processor = WhisperProcessor.from_pretrained(BASE)

def load_manifest(name):
    rows = [json.loads(l) for l in (WORK / 'manifests' / f'{name}.jsonl').read_text().splitlines() if l.strip()]
    for r in rows:
        r['audio_filepath'] = str(WORK / r['audio_filepath'])
    return Dataset.from_list(rows)

train_ds, dev_ds = load_manifest('train'), load_manifest('dev')
print(len(train_ds), 'train clips ·', len(dev_ds), 'dev clips')

# Augmentation (train only, p≈0.3 each) — mimics Sunday variance: PA hiss, echo, pace
from audiomentations import Compose, AddGaussianSNR, TimeStretch, Gain
augments = [AddGaussianSNR(min_snr_db=10.0, max_snr_db=30.0, p=0.3),
            TimeStretch(min_rate=0.9, max_rate=1.1, p=0.3),
            Gain(min_gain_db=-6, max_gain_db=6, p=0.3)]
try:
    from audiomentations import RoomSimulator
    augments.insert(1, RoomSimulator(p=0.3))
except Exception as e:
    print('RoomSimulator unavailable (needs pyroomacoustics), skipping:', e)
augment = Compose(augments)

def featurize(batch, train=False):
    feats, labels = [], []
    for path, text in zip(batch['audio_filepath'], batch['text']):
        audio, _ = sf.read(path, dtype='float32')
        if train:
            audio = augment(samples=audio, sample_rate=SR)
        feats.append(processor.feature_extractor(audio, sampling_rate=SR).input_features[0])
        labels.append(processor.tokenizer(text).input_ids)
    return {'input_features': feats, 'labels': labels}

train_ds.set_transform(lambda b: featurize(b, train=True))
dev_ds.set_transform(lambda b: featurize(b, train=False))

In [ ]:
import torch
from dataclasses import dataclass
from transformers import WhisperForConditionalGeneration
import jiwer

@dataclass
class Collator:
    processor: object
    def __call__(self, features):
        inputs = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(inputs, return_tensors='pt')
        label_feats = [{'input_ids': f['labels']} for f in features]
        labels = self.processor.tokenizer.pad(label_feats, return_tensors='pt')
        lab = labels['input_ids'].masked_fill(labels.attention_mask.ne(1), -100)
        if (lab[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            lab = lab[:, 1:]
        batch['labels'] = lab
        return batch

norm = jiwer.Compose([jiwer.ToLowerCase(), jiwer.RemovePunctuation(), jiwer.RemoveMultipleSpaces(), jiwer.Strip(),
                      jiwer.ReduceToListOfListOfWords()])
def wer(refs, hyps):
    # jiwer ≥3 uses reference_transform; fall back for older versions
    try:
        return jiwer.wer(list(refs), list(hyps), reference_transform=norm, hypothesis_transform=norm)
    except TypeError:
        return jiwer.wer(list(refs), list(hyps), truth_transform=norm, hypothesis_transform=norm)

model = WhisperForConditionalGeneration.from_pretrained(BASE)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None

In [ ]:
# Baseline dev WER for small.en BEFORE training — the Stage C exit check
# (≥25% relative improvement) is measured against this number.
from torch.utils.data import DataLoader

def eval_wer(m):
    m.eval().cuda()
    dl = DataLoader(dev_ds, batch_size=8, collate_fn=Collator(processor))
    hyps, refs = [], []
    with torch.no_grad():
        for batch in dl:
            out = m.generate(input_features=batch['input_features'].cuda().half() if next(m.parameters()).dtype==torch.float16 else batch['input_features'].cuda(), max_length=225)
            hyps += processor.batch_decode(out, skip_special_tokens=True)
            lab = batch['labels'].masked_fill(batch['labels'] == -100, processor.tokenizer.pad_token_id)
            refs += processor.batch_decode(lab, skip_special_tokens=True)
    return wer(refs, hyps), refs, hyps

baseline_wer, _, _ = eval_wer(model)
print(f'baseline small.en dev WER: {baseline_wer:.4f}')

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback

lora = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, target_modules=['q_proj', 'v_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

def compute_metrics(pred):
    ids = pred.predictions
    lab = np.where(pred.label_ids != -100, pred.label_ids, processor.tokenizer.pad_token_id)
    hyps = processor.batch_decode(ids, skip_special_tokens=True)
    refs = processor.batch_decode(lab, skip_special_tokens=True)
    print('\n--- samples ---')
    for r, h in list(zip(refs, hyps))[:3]:
        print(f'  ref: {r}\n  hyp: {h}\n')
    return {'wer': wer(refs, hyps)}

args = Seq2SeqTrainingArguments(
    output_dir='checkpoints',
    learning_rate=1e-3, warmup_ratio=0.1, lr_scheduler_type='linear',
    per_device_train_batch_size=8, gradient_accumulation_steps=2,
    per_device_eval_batch_size=8,
    fp16=True, num_train_epochs=4,
    eval_strategy='epoch', save_strategy='epoch',
    predict_with_generate=True, generation_max_length=225,
    load_best_model_at_end=True, metric_for_best_model='wer', greater_is_better=False,
    logging_steps=25, report_to='none', remove_unused_columns=False, label_names=['labels'],
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=dev_ds,
    data_collator=Collator(processor), compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()

In [ ]:
# Exit check + save adapters
final = trainer.evaluate()
improvement = (baseline_wer - final['eval_wer']) / baseline_wer
print(f"dev WER: {baseline_wer:.4f} → {final['eval_wer']:.4f}  ({improvement:.0%} relative improvement)")
if improvement < 0.25:
    print('✗ exit check: under 25% relative improvement — see CLAUDE.md Troubleshooting before tweaking hyperparameters at random.')
else:
    print('✓ exit check passed')

model.save_pretrained('adapters')
!zip -qr adapters.zip adapters
from google.colab import files
files.download('adapters.zip')
print('unzip into asr-finetune/training/adapters/ on your machine, then run scripts/08_export.py')